# Aralin 12 - Pagbawas ng Kasaysayan ng Chat gamit ang Agent Scratchpad

Ipinapakita ng notebook na ito kung paano pamahalaan ang konteksto sa mahahabang pag-uusap gamit ang Microsoft Agent Framework. Habang lumalaki ang mga pag-uusap, tumataas ang bilang ng token — sa huli ay lalampas sa konteksto ng window ng modelo. Nilulutas namin ito gamit ang **pattern ng pagbubuod ng konteksto** at isang **agent scratchpad** para sa matatag na memorya.

## Ano ang iyong Matututuhan:
1. **Bakit Mahalaga ang Pamamahala ng Konteksto**: Pag-unawa sa mga limitasyon ng token at mga window ng konteksto
2. **Context-Aware Agents**: Paggawa ng mga agent na namamahala ng kanilang sariling konteksto ng pag-uusap
3. **Pattern ng Pagbubuod ng Konteksto**: Paggamit ng mga kasangkapan upang paikliin ang kasaysayan ng pag-uusap
4. **Agent Scratchpad**: Matatag na memorya na nananatili kahit nabawasan ang konteksto

## Mga Kinakailangan:
- Azure OpenAI na naka-setup na may mga environment variables na naka-configure
- Pag-unawa sa mga pangunahing konsepto ng agent mula sa mga nakaraang aralin


## Pagsasaayos


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Bakit Mahalaga ang Pamamahala ng Konteksto

Bawat LLM ay may hangganang **context window** — ang pinakamataas na bilang ng mga token na kaya nitong iproseso sa isang kahilingan. Habang nagpapatuloy ang isang multi-turn na pag-uusap:

- **Lumalaki ng linear ang bilang ng token** sa bawat mensahe ng gumagamit at tugon ng katulong.
- **Nangunguna sa gastos ang mga prompt token** dahil ipinapadala muli ang buong kasaysayan sa bawat turn.
- Sa huli ang pag-uusap ay **lalampas sa context window** at ang modelo ay magpuputol o magbibigay ng error.

### Mga Estratehiya sa Pamamahala ng Konteksto

| Estratehiya | Paano Ito Gumagana | Kapalit |
|---|---|---|
| **Pagtatapyas** | Alisin ang mga pinakalumang mensahe | Nawawala ang maagang konteksto |
| **Pagbubuod** | Pinaiksi ang mga lumang mensahe sa isang buod | May ilang detalye ang nawawala, pero naiingatan ang mahahalagang punto |
| **Scratchpad / Panlabas na Memorya** | Itago ang mahahalagang impormasyon sa labas ng pag-uusap | Nangangailangan ng pagtawag sa tool, pero nananatili kahit bumaba ang detalye |

Sa notebook na ito pinag-sama namin ang **pagbubuod** at isang **scratchpad tool** para mapanatili ng ahente ang pagkakaugnay kahit paikliin ang kasaysayan ng pag-uusap.


## Paglikha ng Isang Ahente na Nakakaalam sa Konteksto


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Pag-simulate ng Isang Mahabang Usapan

Dumaan tayo sa isang multi-turn na usapan upang makita kung paano nag-iipon ang konteksto. Dapat mapanatili ng ahente ang mga mahahalagang detalye (mga gusto, badyet, mga petsa ng paglalakbay) sa bawat turno at ipakita ang tuloy-tuloy na daloy.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pansinin kung paano pinananatili ng ahente ang konteksto mula sa mga naunang pagsasalita — alam nito ang tungkol sa Japan, sushi, mga templo, potograpiya, ang badyet na $3000, paglalakbay nang mag-isa, at ang oras ng Abril. Sa maikling pag-uusap ay mahusay ito, ngunit habang lumalaki ang pag-uusap ay nagiging mahal ang buong kasaysayan upang muling ipadala.

Ipagpatuloy natin ang pag-uusap sa mas maraming bahagi upang makita ang pag-ipon ng konteksto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Pattern ng Pagbubuod ng Konteksto

Habang lumalago ang pag-uusap, maaari nating gamitin ang isang **tool sa pagbubuod** upang paikliin ang naipon na konteksto sa isang compact na porma. Tinatawag ng ahente ang tool na ito upang itala ang mga pangunahing kagustuhan upang kahit tanggalin ang mga lumang mensahe, mapanatili ang mahahalagang impormasyon.

Ang pattern na ito ang pundasyon para sa mas sopistikadong pagbawas ng kasaysayan:
1. Kinilala ng ahente ang mga pangunahing katotohanan mula sa pag-uusap
2. Tinatawag nito ang tool sa pagbubuod upang itala ang mga ito
3. Maaaring ligtas na tanggalin ang mga lumang mensahe dahil nasasaklaw ng buod ang mga mahahalaga

Sa ibaba ay inilalarawan namin ang isang `summarize_preferences` tool na maaaring tawagin ng ahente upang maisulat ang compact na buod ng mga natutunan nito.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Buod

Sa araling ito natutunan mo kung paano pamahalaan ang konteksto sa mga mahabang pag-uusap ng ahente gamit ang Microsoft Agent Framework:

### Pangunahing Mga Konsepto
- **May hangganan ang mga window ng konteksto** — bawat token sa kasaysayan ng pag-uusap ay may halaga at binibilang sa limitasyon.
- **Mga kasangkapang pang-pagsasama-sama** ay nagpapahintulot sa ahente na paikliin ang naipong konteksto sa maikling mga buod, na nagpapabawas sa paggamit ng token habang pinangangalagaan ang mahalagang impormasyon.
- **Panandaliang panulat ng ahente** ay nagbibigay ng permanenteng panlabas na memorya na nananatili kahit pa paikliin ang pag-uusap.

### Ano ang Iyong Naitayo
- Isang **ahente na may kamalayan sa konteksto** na nagpapanatili ng tuloy-tuloy na daloy sa maraming ulit ng pag-uusap
- Isang **kasangkapang pang-pagsasama-sama** (`summarize_preferences`) na nagtatala ng mahahalagang detalye ng gumagamit sa isang maikling format
- Isang **maramihang ulit ng pag-uusap** na nagpapakita ng pagpapanatili ng konteksto at paghawak ng pagbabago

### Mga Aplikasyon sa Tunay na Mundo
- **Mga Bot sa Serbisyo sa Kostumer**: Naaalala ang mga kagustuhan sa mga mahahabang sesyon ng suporta
- **Personal na mga Katulong**: Nagsusubaybay sa mga ongoing na proyekto nang hindi kailangang muling ipaliwanag ang konteksto
- **Mga Guro sa Edukasyon**: Pinapanatili ang progreso ng estudyante sa maraming pakikipag-ugnayan

### Mga Susunod na Hakbang
- Ipatupad ang isang buong kasangkapang panulat na may file-based na pagpapanatili
- Magdagdag ng awtomatikong pagtatanggal ng kasaysayan pagkatapos ng pagsasama-sama
- Pagsamahin sa mga vector database para sa semantic memory search
- Bumuo ng mga ahente na makakapagpatuloy ng pag-uusap ilang araw pagkatapos na may buong konteksto


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
